# 02b. Aplicar Splink (predict + clustering)

Carrega `splink_model.json` treinado no [`02_treinar_splink.ipynb`](02_treinar_splink.ipynb).
Não retreina: se o JSON não existir, falha apontando o notebook de treino.

`predict(0.5)` só gera candidatos. Clustering usa `THRESHOLD_AVALIACAO`
(env, default 0,95). O Linker aplica no **conjunto inteiro**
(`censo_limpo` / `cpf_limpo`); não há amostra de registros.

Próximo: [`03_avaliar.ipynb`](03_avaliar.ipynb) (score/cluster) e
[`04_atribuir.ipynb`](04_atribuir.ipynb) (1 CPF por Censo).


In [ ]:
import json
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

import pandas as pd
from IPython.display import display
from config import (
    DUCKDB_MEMORY_LIMIT,
    DUCKDB_THREADS,
    OUTPUT_DIR,
    PREDICT_NUM_CHUNKS_LEFT,
    PREDICT_NUM_CHUNKS_RIGHT,
    SPLINK_CLUSTERS,
    SPLINK_INPUT_VIEW,
    SPLINK_MODEL_JSON,
    SPLINK_PREDICTIONS,
    TABELA_CENSO_LIMPA,
    TABELA_CPF_LIMPA,
    THRESHOLD_AVALIACAO,
    drop_splink_temp_tables,
    get_connection,
    get_splink_db_api,
    materialize_splink_input,
    print_paths,
    require_tables,
)

print_paths()
if not SPLINK_MODEL_JSON.exists():
    raise RuntimeError(
        f'Modelo Splink não encontrado: {SPLINK_MODEL_JSON}. '
        'Rode notebooks/02_treinar_splink.ipynb para treinar e gravar o JSON. '
        'Este notebook só carrega o modelo — não retreina.'
    )

con = get_connection()
drop_splink_temp_tables(con)
require_tables(con, [TABELA_CENSO_LIMPA, TABELA_CPF_LIMPA], notebook_origem='00b')
materialize_splink_input(con)
db_api = get_splink_db_api(con)

n_reg = con.execute(f'SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW}').fetchone()[0]
duck_settings = con.execute(
    "SELECT current_setting('threads'), current_setting('memory_limit')"
).fetchone()
print(f'Registros: {n_reg:,}')
print(
    f'DuckDB: threads={duck_settings[0]}, memory_limit={duck_settings[1]} '
    f'(defaults: {DUCKDB_THREADS}, {DUCKDB_MEMORY_LIMIT})'
)
print('Modelo:', SPLINK_MODEL_JSON)
print('Threshold (clusters):', THRESHOLD_AVALIACAO)
print('Predict chunks L/R:', PREDICT_NUM_CHUNKS_LEFT, PREDICT_NUM_CHUNKS_RIGHT)


## Linker a partir do JSON

Reconstrói o `Linker` como o 03: `json.load` + `Linker(..., data, db_api=...)`.
Comparisons vêm do JSON; não se redefinem aqui. Views = conjunto inteiro
(`assert` na célula abaixo, antes do predict).


In [ ]:
from splink import Linker

con.execute(f'''
CREATE OR REPLACE VIEW splink_censo AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''')
con.execute(f'''
CREATE OR REPLACE VIEW splink_cpf AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'cpf'
''')
print(
    'splink_censo:', con.execute('SELECT COUNT(*) FROM splink_censo').fetchone()[0],
    '| splink_cpf:', con.execute('SELECT COUNT(*) FROM splink_cpf').fetchone()[0],
)

n_censo_limpo = con.execute(f'SELECT COUNT(*) FROM {TABELA_CENSO_LIMPA}').fetchone()[0]
n_cpf_limpo = con.execute(f'SELECT COUNT(*) FROM {TABELA_CPF_LIMPA}').fetchone()[0]
n_censo_view = con.execute('SELECT COUNT(*) FROM splink_censo').fetchone()[0]
n_cpf_view = con.execute('SELECT COUNT(*) FROM splink_cpf').fetchone()[0]
assert n_censo_view == n_censo_limpo and n_cpf_view == n_cpf_limpo, (
    f'Linker não está no conjunto inteiro: '
    f'censo {n_censo_view} vs limpo {n_censo_limpo}; '
    f'cpf {n_cpf_view} vs limpo {n_cpf_limpo}'
)
print('Linker: conjunto inteiro', f'{n_censo_view:,}', f'{n_cpf_view:,}')

with open(SPLINK_MODEL_JSON, 'r') as file:
    data = json.load(file)

data['retain_intermediate_calculation_columns'] = True

linker = Linker(
    ['splink_censo', 'splink_cpf'],
    data,
    db_api=db_api,
    input_table_aliases=['censo', 'cpf'],
)


## Predict + clustering

`predict(0.5)` só gera candidatos. Clustering e validação usam
`THRESHOLD_AVALIACAO` (env, default 0,95).

Resultados ficam como `SplinkDataFrame` (tabela DuckDB). Export usa
`to_parquet` sem materializar tudo em pandas. Diagnósticos e waterfall
usam amostra (`limit` / SQL). Em volume grande, ajuste
`PREDICT_NUM_CHUNKS_LEFT` / `PREDICT_NUM_CHUNKS_RIGHT` no `config`.


In [ ]:
predict_kwargs = {'threshold_match_probability': 0.5}
if PREDICT_NUM_CHUNKS_LEFT is not None:
    predict_kwargs['num_chunks_left'] = PREDICT_NUM_CHUNKS_LEFT
if PREDICT_NUM_CHUNKS_RIGHT is not None:
    predict_kwargs['num_chunks_right'] = PREDICT_NUM_CHUNKS_RIGHT

df_predict = linker.inference.predict(**predict_kwargs)
print('predictions table:', df_predict.physical_name)
print('pares (aprox):', con.execute(f'SELECT COUNT(*) FROM {df_predict.physical_name}').fetchone()[0])


In [ ]:
print('Threshold (clusters):', THRESHOLD_AVALIACAO)
clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    df_predict, threshold_match_probability=THRESHOLD_AVALIACAO,
)
print('clusters table:', clusters.physical_name)
n_pares = con.execute(f'SELECT COUNT(*) FROM {df_predict.physical_name}').fetchone()[0]
n_clust = con.execute(
    f'SELECT COUNT(DISTINCT cluster_id) FROM {clusters.physical_name}'
).fetchone()[0]
print('Pares:', n_pares, 'Clusters:', n_clust)


## Waterfall — amostra de pares

Contribuição de cada comparação ao match weight (5 pares).


In [ ]:
# Amostra pequena — não carrega o parquet inteiro em memória.
records_to_plot = df_predict.as_record_dict(limit=5)
linker.visualisations.waterfall_chart(records_to_plot, filter_nulls=False)


In [ ]:
pd.set_option('display.max_columns', None)
display(
    con.execute(f'''
    SELECT *
    FROM {df_predict.physical_name}
    WHERE CAST(unique_id_l AS VARCHAR) LIKE '%censo%'
      AND CAST(unique_id_r AS VARCHAR) LIKE '%cpf%'
    LIMIT 5
    ''').df()
)


## Cluster studio

Dashboard interativo para inspecionar clusters (amostra por tamanho).


In [ ]:
from IPython.display import IFrame, display

CLUSTER_STUDIO_HTML = OUTPUT_DIR / 'dashboards' / 'cluster_studio.html'
CLUSTER_STUDIO_HTML.parent.mkdir(parents=True, exist_ok=True)

linker.visualisations.cluster_studio_dashboard(
    df_predict,
    clusters,
    str(CLUSTER_STUDIO_HTML),
    sampling_method='by_cluster_size',
    overwrite=True,
)
print('Dashboard:', CLUSTER_STUDIO_HTML)
display(IFrame(src=str(CLUSTER_STUDIO_HTML), width='100%', height=1200))


## Diagnóstico dos scores (sem rótulos)

Distribuição de match weight e tamanho de cluster. Avaliação no
[`03_avaliar.ipynb`](03_avaliar.ipynb); lista 1 CPF por Censo no
[`04_atribuir.ipynb`](04_atribuir.ipynb). Cheque os match weights: discordar nome/DOB
deve penalizar vários bits, não ≈ 0.


In [ ]:
display(
    con.execute(f'''
    SELECT
        COUNT(*) AS n,
        MIN(match_probability) AS min_p,
        AVG(match_probability) AS avg_p,
        MAX(match_probability) AS max_p,
        MIN(match_weight) AS min_w,
        AVG(match_weight) AS avg_w,
        MAX(match_weight) AS max_w
    FROM {df_predict.physical_name}
    ''').df()
)
# DuckDB 1.x (Singed) não tem width_bucket. 20 faixas de 3 bits em [-20, 40].
display(
    con.execute(f'''
    SELECT
        faixa,
        CASE WHEN faixa <= 0 THEN NULL ELSE -20 + 3 * (faixa - 1) END AS peso_min,
        CASE WHEN faixa >= 21 THEN NULL ELSE -20 + 3 * faixa END AS peso_max,
        n_pares
    FROM (
        SELECT
            CASE
                WHEN match_weight < -20 THEN 0
                WHEN match_weight > 40 THEN 21
                WHEN match_weight = 40 THEN 20
                ELSE CAST(FLOOR((match_weight + 20) / 3.0) AS INTEGER) + 1
            END AS faixa,
            COUNT(*) AS n_pares
        FROM {df_predict.physical_name}
        WHERE match_weight IS NOT NULL
        GROUP BY 1
    )
    ORDER BY 1
    ''').df()
)


In [ ]:
tamanhos = con.execute(f'''
SELECT cluster_id, COUNT(*) AS tamanho
FROM {clusters.physical_name}
GROUP BY 1
''').df()
print(
    f"Clusters: {len(tamanhos):,} | maior: {tamanhos['tamanho'].max():,} | "
    f"singletons: {(tamanhos['tamanho'] == 1).sum():,}"
)
display(
    tamanhos['tamanho'].value_counts().sort_index().head(20)
    .rename('n_clusters').to_frame().rename_axis('tamanho')
)
display(
    tamanhos.sort_values('tamanho', ascending=False).head(10)
    .set_index('cluster_id')
)


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_predict.to_parquet(str(SPLINK_PREDICTIONS), overwrite=True)
clusters.to_parquet(str(SPLINK_CLUSTERS), overwrite=True)
print('Predictions:', SPLINK_PREDICTIONS)
print('Clusters:', SPLINK_CLUSTERS)


## Encerrar

Artefatos prontos para o [`03_avaliar.ipynb`](03_avaliar.ipynb) e o
[`04_atribuir.ipynb`](04_atribuir.ipynb).


In [ ]:
for label, p in [
    ('modelo', SPLINK_MODEL_JSON),
    ('predictions', SPLINK_PREDICTIONS),
    ('clusters', SPLINK_CLUSTERS),
]:
    print(f'{label:12s} {"ok " if p.exists() else "FALTA"} {p}')


In [ ]:
con.close()
